In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE", "AV_DRIVE/training"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, AV, resize_to=1024, output_dir="tmp/dataset-test")

Found 187 branch digraphs...


Processing samples:   0%|          | 0/187 [00:00<?, ?it/s]

In [3]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")


Processing...
Done!


## Graph Augment


In [8]:
ID = 0
m, digraph, _ = dataset.jppype_show(ID, augment=True)
m

[ WARN:0@146.500] global loadsave.cpp:1671 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">001_G/vascx</h3>'), HTML(value='<h3 style="text-…

In [5]:
dataset.get(20, augment=True, version="fvt")
%timeit dataset.get(20, augment=True, version="fvt")

289 ms ± 3.58 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
subset = dataset.split(list(range(20))).preload()
%timeit [subset.get(i, augment=True, version="fvt") for i in range(3)]

Preloading dataset: 100%|██████████| 20/20 [00:01<00:00, 15.00it/s]


409 ms ± 10.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
from fundus_vessels_toolkit.utils.profiling import Profiler, watch

with Profiler():
    for i in range(len(subset)):
        subset.get(i, augment=True, version="fvt")

In [17]:
subset.get(0, augment=True, version="fvt")
%timeit subset.get(0, augment=True, version='fvt')

121 ms ± 1.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
import cProfile

dataset.preload()

In [ ]:
cProfile.run("dataset.get(20, augment=True, version='fvt')", sort="cumulative")

In [ ]:
from fundus_vessels_toolkit.utils.profiling import ProfilerWatch

ProfilerWatch.reset()
dataset.get(20, augment=True, version="fvt")
print(ProfilerWatch.get("split_branch").print())

In [5]:
from fundus_toolkits.transform import ElasticTransform, ElasticTransformLegacy

data = dataset.get(20, augment=False, version="fvt")

elastic = ElasticTransform.random(data.img.shape[-2:], 80, 200)
elastic_leg = ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)


%timeit ElasticTransform.random(data.img.shape[-2:], 80, 200)
%timeit ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)

597 μs ± 11.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.4 ms ± 30.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [11]:
sample = dataset.get_sample(20)

In [ ]:
%timeit sample.graphes['fvt'].transform(elastic)
%timeit sample.graphes['fvt'].transform(elastic_leg)

563 ms ± 9.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
26.3 ms ± 1.4 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
%timeit elastic.warp(data.img.permute(1, 2, 0))
%timeit elastic_leg.warp(data.img.permute(1, 2, 0).numpy())

26.1 ms ± 751 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
36 ms ± 384 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
from fundus_vessels_toolkit.utils.profiling import watch, ProfilerWatch

elastic = ElasticTransform.random(data.img.shape[-2:], 80, 200)
elastic_leg = ElasticTransformLegacy.random(data.img.shape[-2:], 80, 200)
ProfilerWatch.reset()
elastic.warp(data.img.permute(1, 2, 0))
elastic_leg.warp(data.img.permute(1, 2, 0).numpy())
sample.graphes["fvt"].transform(elastic)
sample.graphes["fvt"].transform(elastic_leg)
print(watch("warp").print())
print(watch("ElasticTransformLegacy._warp").print())
print(watch("ElasticTransform._transform inverse").print())
print(watch("ElasticTransformLegacy._transform inverse").print())

warp                 18.21ms (runs=1)
├── grid_indices     370.5µs (runs=1)
├── transform         7.61ms (runs=1)
├── grid_sample      10.21ms (runs=1)
└── to numpy           6.9µs (runs=1)
ElasticTransformLegacy._warp         34.32ms (runs=1)
├── grid_indices                      3.85ms (runs=1)
├── transform                        14.72ms (runs=1)
└── remap                            15.73ms (runs=1)
ElasticTransform._transform inverse         10.91ms (runs=533, avg=  20.5µs)
└── grid_indices                             73.2µs (runs=533, avg=   0.1µs)
ElasticTransformLegacy._transform inverse          9.35ms (runs=533, avg=  17.5µs)
├── generate grid                                  1.34ms (runs=533, avg=   2.5µs)
└── compute inverse displacement                   6.89ms (runs=533, avg=  12.9µs)
